#TCC em GenAI na UFG

Este notebook representaa a parte prática do TCC em GenAI na UFG em 2026.

Título: Orquestração de Modelos de Linguagem de Grande Escala (LLMs) para Produção de Respostas Jurídicas: um Estudo Baseado em Avaliação Cruzada entre Modelos

Integrantes:
- CÁSSIO OLIVEIRA CAMILO
- CHARLY BRAGA VENTURA
- MÁRCIO MEIRA E SILVA


# Inicializa

In [ ]:
import time
start_all = time.perf_counter() #usado para medir o tempo total da execução

## Define memória do LangGraph para gravar resultados

In [ ]:
from typing import Annotated, TypedDict, List
import operator

class GraphState(TypedDict):
    question: str
    responses: Annotated[List[dict], operator.add]  #Exemplo: [{modelo: "Gemini", resposta: "..."}]
    evaluations: Annotated[List[dict], operator.add] # Exemplo: [{avaliador: "Llama", feedback: "..."}]
    identity_map: dict
    final_consensus: List[dict]

#Carrega modelos de IA

## Instala e carrega modelos *open-weight* pelo Ollama

In [ ]:
%%capture
!pip install langchain_community

In [ ]:
from langchain_community.chat_models import ChatOllama
import re

def get_model_data(fullname, nickname):
  name = re.split('[:|-]', fullname)[0]
  if "gpt" in name:
    model_instance = ChatOpenAI(model=fullname) #gpt
  elif "gemini" in name:
    model_instance = ChatGoogleGenerativeAI(model=fullname) #Gemini
  else:
    model_instance = ChatOllama(model=fullname) #Ollama

  model_data = {'model': model_instance,
                'name': name,
                'fullname': fullname,
                'nickname': nickname}
  return model_data

#Adiciona modelos na lista
model_list = []
model_list.append(get_model_data('deepseek-r1:8b','A'))
model_list.append(get_model_data('mistral-small3.1:24b','B'))
model_list.append(get_model_data('llama4:16x17b','C'))

/tmp/ipykernel_1648/1498162302.py:11: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  model_instance = ChatOllama(model=fullname) #Ollama


In [ ]:
# 1. Instala dependências de sistema
!apt-get update && apt-get install -y zstd

# 2. Instala o Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Inicia o servidor em background
import subprocess
import time
import os

print("Starting Ollama server...")
#Redireciona o log para um arquivo para não poluir o console
with open("ollama_log.txt", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

time.sleep(10) # Tempo para o servidor estabilizar

# 4. Baixa modelos abertos
print('Downloaded ' + str(len(model_list)) + ' models ...')
for model_data in model_list:
  model_fullname = model_data['fullname']
  !ollama pull $model_fullname

print("Done!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,245 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,293 kB]
Get:13 http://archive.ubuntu.com/ubun

## Carrega modelos **pagos**

###Carrega Gemini

In [ ]:
!pip install langchain-google-genai

import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

#Aqui se carrega a API Gemini no Colab (deve-se configurar a API no AI Studio)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

#Adiciona modelo na lista
model_list.append(get_model_data('gemini-3.1-flash-lite-preview','D'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 6.5 MB/s eta 0:00:00


TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

### Carrega GPT

In [ ]:
!pip install langchain_openai

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('GPT_API_KEY')

#Adiciona modelo na lista
model_list.append(get_model_data('gpt-5-nano','E'))

###Carrega Chairman

In [ ]:
 #Esta temperature resulta em modelo mais determinítico para ranquear os modelos baseado em notas de avaliações em pares
chairman = ChatOpenAI(model = "gpt-5.4", temperature = 0)

###

#Define padrão de respostas, avaliações e *rankings*

## Métodos para respostas e avaliações

In [ ]:
def format_seg_to_hhmmss(seconds):
    # Formata o tempo total de segundos para horas, minutos e segundos
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    if hours > 0:
        return f"{hours}h {minutes}m {secs:.2f}s"
    elif minutes > 0:
        return f"{minutes}m {secs:.2f}s"
    else:
        return f"{secs:.2f}s"

In [ ]:
#Retorna metadata padrão, independente do modelo utilizado
def get_metadata(res, name):
  if 'gemini' in name:
    return res.usage_metadata #gemini
  else:
    return res.response_metadata #gpt, deepseek, llhama e mistral

### Define métodos para respostas de cada modelo

In [ ]:
def call_answer(state: GraphState, model, name, nickname):
    start = time.perf_counter()
    instruction_question =  f"""
      Você é um Especialista em Direito Administrativo.
      Responda à questão técnica abaixo seguindo rigorosamente estas premissas:

      1. **Objetividade:** Vá direto ao ponto. Comece respondendo claramente à pergunta (Sim/Não/Depende).
      2. **Fundamentação:** Cite obrigatoriamente o dispositivo legal ou jurisprudência relevante (ex: Lei 14.133, Lei 8.429 com as alterações da 14.230, Constituição Federal, etc.).
      3. **Extensão:** Limite sua resposta a no máximo 2 ou 3 parágrafos curtos.
      4. **Atualidade:** Baseie-se no estado atual da legislação brasileira (2026).

      ### QUESTÃO:
      {state["question"]}
      """
    res = model.invoke(instruction_question)

    end = time.perf_counter()
    latency = format_seg_to_hhmmss(end - start)

    metadata = get_metadata(res, name)

    return {"responses": [{"name": name,
                           "nickname" : nickname,
                           "answer": res.content,
                           "latency": latency,
                           "metadata": metadata }]}

### Define métodos para avaliações em pares

In [ ]:
def get_evaluation_prompt(answer_model, question):
    context = f"Modelo {answer_model['nickname']}: {answer_model['answer']}"
    eval = f"""
      Atue como Avaliador Jurídico Sênior em um sistema de "Double-Blind Peer Review".
      Sua tarefa é avaliar a resposta do {answer_model['nickname']} para a pergunta: '{question}'

      ### RESPOSTA A SER AVALIADA:
      {context}

      ### DIRETRIZES DE AVALIAÇÃO (Seja extremamente conciso):
      1. Coerência (30%): A lógica jurídica faz sentido?
      2. Aderência (30%): Respeita a norma/jurisprudência?
      3. Alucinação (20%): Inventou leis ou fatos?
      4. Clareza (10%): Texto compreensível?
      5. Humanização (10%): Tom adequado?

      ### FORMATO OBRIGATÓRIO DE RESPOSTA (Siga rigorosamente):
      ID_MODELO: {answer_model['nickname']}
      NOTAS_CRITERIOS:
      - C1 (Coerência): [Nota 0-10] | [Frase curta de justificativa]
      - C2 (Aderência): [Nota 0-10] | [Frase curta de justificativa]
      - C3 (Alucinação): [Nota 0-10] | [Frase curta de justificativa]
      - C4 (Clareza): [Nota 0-10] | [Frase curta de justificativa]
      - C5 (Humanização): [Nota 0-10] | [Frase curta de justificativa]

      NOTA_FINAL: [Cálculo ponderado de 0 a 10]
      VEREDITO_CURTO: [Máximo 20 palavras resumindo o desempenho]

      Responda em Português Brasileiro. Não escreva introduções ou conclusões.
      """
    return eval

def call_evaluate(state: GraphState, model, evaluator_name, evaluator_nickname, answer_model_name):
    start = time.perf_counter()
    answer_model = next(
        (r for r in state["responses"] if r['name'] == answer_model_name),
        None
    )

    if not answer_model:
        return {"evaluations": []}

    prompt = get_evaluation_prompt(answer_model, state["question"])

    res = model.invoke(prompt)

    end = time.perf_counter()
    latency = format_seg_to_hhmmss(end - start)

    metadata = get_metadata(res, evaluator_name)

    return {"evaluations": [{
        "evaluator_name": evaluator_name,
        "evaluator_nickname": evaluator_nickname,
        "feedback": res.content,
        'answer_model_name': answer_model_name,
        "latency": latency,
        "metadata": metadata}]}

### Define como *Chairman* sumariza os resutaldos e realiza *rankings*

In [ ]:
def final_synthesis(state: GraphState):
  start = time.perf_counter()
  evals = "\n".join([f"Veredito do {e['evaluator_nickname']}:\n{e['feedback']}" for e in state["evaluations"]])
  prompt_chairman = f"""
    Você é o Presidente da Banca Examinadora. Sua função é consolidar as avaliações de múltiplos juízes e definir o veredito final.

    ### AVALIAÇÕES RECEBIDAS:
    {evals}

    ### INSTRUÇÕES DE PONDERAÇÃO:
    1. **Normalização:** Extraia as notas finais de cada juiz para cada modelo.
    2. **Penalização Crítica:** Se algum juiz apontar Erro Crítico de Lei (ex: citar lei revogada), esse modelo deve ser penalizado no ranking, independentemente da clareza do texto.
    3. **Resolução de Divergência:** Se os juízes discordarem, priorize o argumento que for mais específico juridicamente (citação de artigos e jurisprudência).
    4. **Cálculo do Ranking:** O 1º lugar deve ser o que obteve a maior média ponderada. O último lugar o que teve falhas técnicas graves.

    ### TAREFA:
    1. Analise cuidadosamente os pontos positivos e negativos de cada modelo mencionados pelos juízes.
    2. Determine quem teve o melhor desempenho técnico e quem teve o pior.
    3. Resuma a conclusão consensual em um parágrafo curto.
    4. Gere o RANKING FINAL seguindo rigorosamente esta ordem:
      - O 1º lugar DEVE ser o modelo com a melhor resposta.
      - O último lugar DEVE ser o modelo com a pior resposta.

    ### FORMATO DE RESPOSTA:
    - Conclusão: [Seu resumo aqui]
    - Ranking:
      1. [Nome do Modelo] - (Melhor) Justificativa breve.
      2. [Nome do Modelo] - Justificativa breve.
      ...
      N. [Nome do Modelo] - (Pior) Justificativa breve.

    Pense passo a passo antes de escrever o ranking para garantir que o melhor esteja no topo. Responda em Português Brasileiro.
    """
  res = chairman.invoke(prompt_chairman) # Chairman que fornece os rankings com base nas avaliações  em pares
  end = time.perf_counter()

  latency = format_seg_to_hhmmss(end - start)

  metadata = res.response_metadata

  return {"final_consensus": {"output": res.content,
                             "latency": latency,
                             "metadata": metadata}}


# Workflow


Cria-se um grafo (ou *Worflow*) com ajuda com LangGraph para definir como serão as interações entre modelos respondentes, avaliadores e *Chairman*.



In [ ]:
from functools import partial
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(GraphState)

eval_nodes = []

for model_data in model_list:
    node_answer = "answer_" + model_data['name']
    partial_answer = partial(call_answer,
                             model = model_data['model'],
                             name = model_data['name'],
                             nickname = model_data['nickname'])

    workflow.add_node(node_answer,  partial_answer)

    workflow.add_edge(START, node_answer)

    for eval_data in model_list:
        if model_data['name'] != eval_data['name']:
            node_eval = f"eval_{eval_data['name']}_of_answer_{model_data['name']}"
            eval_nodes.append(node_eval) # Add to the global tracker

            partial_evaluate =  partial(call_evaluate,
                                        model = eval_data['model'],
                                        evaluator_name =  eval_data['name'],
                                        evaluator_nickname = eval_data['nickname'],
                                        answer_model_name = model_data['name'])

            workflow.add_node(node_eval, partial_evaluate)

            workflow.add_edge(node_answer, node_eval)

# CHAIRMAN
workflow.add_node("synthesis", final_synthesis)

# Conecta todos avaliadores ao Chairman
for eval_node in eval_nodes:
    workflow.add_edge(eval_node, "synthesis")

workflow.add_edge("synthesis", END)

app = workflow.compile()

# Main

## Define as perguntas

> Add blockquote



In [ ]:
inputs = []

#Questão 1
inputs.append ( {
    "question": "A penhora dos bens de autarquias e sociedades de economia mista, entidades administrativas integrantes da Administração Indireta é possível?",
    "responses": [],
    "evaluations": []
  }
)

#Questão 2
inputs.append ( {
    "question": "Quando a Administração adota nova interpretação de norma com conteúdo indeterminado, ela pode impor novo dever ou restrição ao administrado? ",
    "responses": [],
    "evaluations": []
  }
)

#Questão 3
inputs.append ( {
    "question": "O contrato administrativo celebrado com a administração, sob a égide da Lei nº 14.133/2021, que não prevê expressamente a utilização de meios alternativos de solução de controvérsias, pode ser aditado para admitir instrumentos consensuais para tal finalidade? ",
    "responses": [],
    "evaluations": []
  }
)

#Questão 4
inputs.append ( {
    "question": "São insuscetíveis de desapropriação para fins de reforma agrária a pequena e a média propriedade rural, desde que seu proprietário não possua outra?",
    "responses": [],
    "evaluations": []
  }
)

#Questão 5
inputs.append ( {
    "question": "Quando houver dano ao erário, a responsabilização por improbidade independe de conduta dolosa? Isto é, o prejuízo patrimonial da administração causado por culpa (negligência, imprudência ou imperícia) do servidor caracteriza improbidade?",
    "responses": [],
    "evaluations": []
  }
)


###

## Identifica os modelos, com nomes e apelidos correspondentes

In [ ]:
print("Chairman: ")
print(chairman)
print("\n\nModelos para respostas e avaliações individuais:")
for model_data in model_list:
  print("Nome: ", model_data['name'].split("-|:")[0], " - Apelido: ", model_data['nickname'])

## Executa as ações de respostas e avaliações das LLMs

###Retorna resultados com todas as respostas, avaliações e rankings para todos as questões

In [ ]:
def get_results():
  results = []
  for inp in inputs:
    results.append(app.invoke(inp))
  return results

###Formata o tempo total em nanosegundo para hora, minutos e segundos

In [ ]:
def format_nanoseg_to_hhmmss(nanos):
    total_seconds = nanos / 1_000_000_000
    hours = int(total_seconds// 3600)
    minutes = int((total_seconds % 3600) // 60)
    secs = total_seconds % 60
    if hours > 0:
        return f"{hours}h {minutes}m {secs:.2f}s"
    elif minutes > 0:
        return f"{minutes}m {secs:.2f}s"
    else:
        return f"{secs:.2f}s"

###Imprime os resultados de forma estruturada

In [ ]:
#------------------------------ Respostas ------------------------------

def print_result(results):
  print('+++ RESPOSTAS +++')
  for i, result in enumerate(results):
    print(f"\n+++ Questão {i+1} +++", end="\n\n")
    print((result['question']).lstrip(), end="\n\n")
    responses = result["responses"]
    for j, answer_model in enumerate(responses):
      if j > 0: print('-'*25)
      print("\n+++ Resposta do modelo " + answer_model["name"].capitalize() + f" para Q{i+1} +++", end="\n\n")
      if 'gemini' in answer_model["name"]:
        answer_gemini = answer_model["answer"][0]['text']
        print(answer_gemini.lstrip(), end='\n\n')
      else:
        print((answer_model["answer"]).lstrip(), end='\n\n')

      print_metadata(answer_model)

    print('='*70)

    # --------------------------- Avaliações --------------------------

    print('+++ AVALIAÇÕES INDIVIDUAIS +++')
    print(f"\n+++ Questão {i+1} +++", end="\n\n")
    #print((result['question']).lstrip(),end="\n\n")
    evaluations = result["evaluations"]
    for j, evaluator_model in enumerate(evaluations):
      if j > 0: print('-'*25)
      print(f"\n+++ Feedback do avaliador {evaluator_model["evaluator_name"].capitalize()} sobre o resposta de {evaluator_model['answer_model_name'].capitalize()} para Q{i+1} +++", end="\n\n")
      if 'gemini' in evaluator_model["evaluator_name"]:
        feedback_gemini = evaluator_model["feedback"][0]['text']
        print(feedback_gemini.lstrip(),end='\n\n')
        evaluator_model["feedback"]
      else:
        print((evaluator_model["feedback"]).lstrip(), end='\n\n')
    print('='*70)

    #-------------------------- Chairman ---------------------------------

    print('+++ RANK DO CHAIRMAN +++')
    print(f"\n+++ Questão {i+1} +++",end="\n\n")
    result_chairman = result["final_consensus"]
    print(result_chairman["output"].lstrip())
    print('='*70)
    print('='*70)

## Round 1

In [ ]:
start_round1 = time.perf_counter()
results_round1 = get_results()
print("+++ ROUND 1 +++ \n\n")
print_result (results_round1)

In [ ]:
print('+++ RESPOSTAS +++')
for i, result in enumerate(results_round1):
  print(result)

In [ ]:
end_round1 = time.perf_counter()
print("Tempo total para executar completamente o Round 1:")
format_seg_to_hhmmss(end_round1 - start_round1)

## Round 2

In [ ]:
start_round2 = time.perf_counter()
results_round2 = get_results()
print("+++ ROUND 2 +++ \n\n")
print_result (results_round2)

In [ ]:
end_round2 = time.perf_counter()
print("Tempo total para executar completamente o Round 2:")
format_seg_to_hhmmss(end_round2 - start_round2)

## Tempo de execução total da aplicação

In [ ]:
end_all = time.perf_counter()
print("Tempo total da aplicação rodando, desde carregamento dos modelos até execução dos rounds:")
format_seg_to_hhmmss(end_all - start_all)